In [52]:
from pathlib import Path

import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

from plant_shape_analysis import PlantSequencesDataset, LeafSequencesDataset
from plant_shape_analysis.vis.plot_functions import visualize_plant_sequence, visualize_leaf_sequence

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Plant sequences

In [32]:
dataset_path = Path("../data/TrackPlant3D/versions")
plant_dataset = PlantSequencesDataset(
    dataset_path,
    use_ply=True,
    alignment_method="stem_based",
    estimate_normals=True,
    auto_download=False
)

Applying STEM_BASED alignment to 43 plant sequences...
STEM_BASED alignment complete. 43 sequences aligned.
Estimating normals for all sparse plants in the dataset...
Normal estimation complete.


Plot one plant example with normals

In [36]:
sample_scan = plant_dataset[0]["timepoints"][0]
points = sample_scan["points"]
normals = sample_scan["normals"]

# Plot
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.normals = o3d.utility.Vector3dVector(normals)
# pcd.normalize_normals() # they're already normalized
scale = 0.5  # Adjust this number — smaller = shorter, larger = longer
pcd.normals = o3d.utility.Vector3dVector(
    [n * scale for n in pcd.normals]
)
o3d.visualization.draw_geometries(
        [pcd],
        point_show_normal=True,
        up=[0, 0, 1],
        front=[1, 0, 0],
        lookat=[0, 0, 0],
    )

Visualize a whole sequence with normals

In [33]:
selected_sequences = [
"maize_control_plant1",
"maize_control_plant2",
"maize_control_plant3",
"sorghum_control_plant1",
"sorghum_highlight_plant2",
"sorghum_highlight_plant3",
"tobacco_control_plant1",
"tobacco_control_plant2",
"tobacco_shade_plant3",
"tomato1_control2_plant2",
"tomato1_heat_plant3",
"tomato2_control_plant3"
]

In [39]:
for plant_ts in plant_dataset:
    if plant_ts["sequence_name"] in selected_sequences:
        visualize_plant_sequence(plant_ts, window_name=f"{plant_ts['sequence_name']}", spacing=100, normals_scale=0.2)
    break

### Compute normals from scratch and examine orientation

In [40]:
dataset_path = Path("../data/TrackPlant3D/versions")
plant_dataset = PlantSequencesDataset(
    dataset_path,
    use_ply=True,
    alignment_method="stem_based",
    estimate_normals=False,
    auto_download=False
)

Applying STEM_BASED alignment to 43 plant sequences...
STEM_BASED alignment complete. 43 sequences aligned.


In [44]:
sample_seq = plant_dataset.get_timeseries_by_sequence_name("maize_control_plant1")
sample_seq["timepoints"]

[{'day': 0,
  'points': array([[  6.528425,  -4.4796  ,   3.937625],
         [ -9.05365 ,   5.2608  , -28.6408  ],
         [ -6.94925 ,   7.4867  ,  28.674999],
         ...,
         [ -2.712192,   6.730792,   5.678608],
         [ -7.51203 ,   5.26555 , -22.114559],
         [ -5.865   ,   1.13345 ,  -2.873725]], shape=(10000, 3)),
  'labels': array([1, 0, 2, ..., 2, 0, 1], shape=(10000,)),
  'dense_points': array([[ -7.341084,   6.272159, -26.860611],
         [ -7.410384,   6.271059, -26.871011],
         [ -7.468284,   6.248459, -26.902211],
         ...,
         [ -7.992784,   3.331459, -28.604611],
         [ -7.900084,   3.327159, -28.634811],
         [ -7.804984,   3.317159, -28.661911]], shape=(230348, 3)),
  'dense_labels': array([0, 0, 0, ..., 0, 0, 0], shape=(230348,)),
  'leaf_tip_idxs': array([   2, 9506]),
  'file_path': PosixPath('../data/TrackPlant3D/versions/v1/gt_corrected_v1/maize/1_maize_control_plant1_D00.ply')},
 {'day': 1,
  'points': array([[ -6.83113881, 

In [148]:
leaf_dataset = LeafSequencesDataset(
    dataset_path,
    use_ply=True,
    plant_alignment_method="stem_based",
    alignment_method="pca",
    estimate_plant_normals=True,
    auto_download=False
)

Applying STEM_BASED alignment to 43 plant sequences...
STEM_BASED alignment complete. 43 sequences aligned.
Estimating normals for all sparse plants in the dataset...
Normal estimation complete.
Applying PCA alignment to 218 leaf sequences...
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping

In [153]:
# TODO 1: fix vis needing transformations key-value pair of leafs
# TODO 2: fix last timestep which is upside down 
sample_seq = leaf_dataset.get_timeseries_by_sequence_name("maize_control_plant1_leaf3") 

In [154]:
visualize_leaf_sequence(sample_seq)

In [105]:
sample_seq

{'sequence_name': 'maize_control_plant1_leaf2',
 'plant_sequence_name': 'maize_control_plant1',
 'leaf_id': 2,
 'timepoints': [{'day': 0,
   'leaf_id': 2,
   'points': array([[-6.94925 ,  7.4867  , 28.674999],
          [-6.0848  ,  7.6452  , 10.134026],
          [-4.90621 ,  9.86796 , 19.50724 ],
          ...,
          [-4.86913 ,  8.25408 ,  8.102461],
          [-2.657316,  7.332367,  8.879733],
          [-2.712192,  6.730792,  5.678608]], shape=(1835, 3)),
   'normals': array([[ 0.26048069, -0.96225035,  0.07889282],
          [-0.71553215,  0.69409113, -0.07906485],
          [-0.52373536,  0.82467714, -0.21356238],
          ...,
          [-0.04950897,  0.98913296, -0.13843719],
          [ 0.66421937,  0.73483102, -0.13724431],
          [ 0.7859715 ,  0.59176989, -0.17904526]], shape=(1835, 3)),
   'dense_points': array([[-6.530284,  6.892059, 28.319889],
          [-6.421584,  7.187659, 28.434589],
          [-6.465884,  7.142159, 28.385189],
          ...,
          [-6.

In [142]:
aligned_timepoints, _ = leaf_dataset.align_leaf_sequence(sample_seq)
aligned_seq = {
    "timepoints": aligned_timepoints,
    "sequence_name": sample_seq["sequence_name"],
    "leaf_id": sample_seq["leaf_id"]
}
visualize_leaf_sequence(aligned_seq, show_coordinate_frame=True)

In [ ]:
# It seems this works but only on unaligned leaves
aligned_timepoints_consistent = leaf_dataset._enforce_temporal_normal_consistency(aligned_seq["timepoints"])
aligned_seq_consistent = {
    "timepoints": aligned_timepoints_consistent,
    "sequence_name": sample_seq["sequence_name"],
    "leaf_id": sample_seq["leaf_id"]
}
visualize_leaf_sequence(aligned_seq_consistent, show_coordinate_frame=True)

In [121]:
len(aligned_timepoints_consistent)

9

In [107]:
def estimate_normals(sample_seq):
    for timepoint in sample_seq["timepoints"]:
        points = timepoint["points"]
        if points is not None and len(points) > 0:
            # Estimate normals using Open3D
            pcd = o3d.geometry.PointCloud()
            pcd.points = o3d.utility.Vector3dVector(points)
            pcd.estimate_normals()
            pcd.orient_normals_to_align_with_direction()
            pcd.orient_normals_consistent_tangent_plane(k=30)
            normals = np.asarray(pcd.normals)
            timepoint["normals"] = normals

# Estimate normals for each timepoint
#estimate_normals(sample_seq)

# Ensure temporal consistency of normals across timepoints
#leaf_dataset._enforce_temporal_normal_consistency(sample_seq["timepoints"])

# Visualize
visualize_leaf_sequence(sample_seq)